## Launching model on dataset and collecting output

### Imports

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

from langchain_core.prompts import ChatPromptTemplate
from services.common.llm_interface import LLMInterface
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import os
import torch
import sys

sys.path.append("../../../services/")
from index import Index
from services.experiment.cot.data_process_utils import retrieve_answer_token_index

torch.random.manual_seed(42)

/workspace/Diplom/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")

### Preparation

In [ ]:
ITERATIONS_COUNT = 12000

In [ ]:
torch.random.manual_seed(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
print(device)
print(torch.cuda.get_device_properties(device))

cuda:0
_CudaDeviceProperties(name='NVIDIA L40', major=8, minor=9, total_memory=45385MB, multi_processor_count=142, uuid=441e6008-cd27-34b5-c81a-17dad9d4a894, L2_cache_size=96MB)


In [4]:
dataset = load_dataset("ehovy/race", "high")['train']

In [5]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    attn_implementation="eager",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = model.to(device)
model.eval()

Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 91.96it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [7]:
chat_llm = LLMInterface(
    model=model,
    tokenizer=tokenizer,
    device=device
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert in answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:

            1. Read the question and given options.
            2. Begin a short reasoning process before giving answer and surround your reasoning with <think> and </think> tags.
            3. After the chain of thoughts is complete, select the correct option number between 0 and 3.

            Follow this output format:
            <think>
            SHORT_CHAIN_OF_THOUGHTS_TEXT
            </think>
            Answer: OPTION_NUMBER
            
            Example:
            <think>
            The capital of France is Paris, which is option 2.
            </think>
            Answer: 2 

            Now answer the following question following output format.
            """,
        ),
        (
            "human",
            """
            Question: 
            {input_question}
            
            Options:
            {input_options}
            """
        )
    ]
)

llm_chain = prompt | chat_llm

In [8]:
len(dataset)

62445

### Running model on dataset and collecting results


In [9]:
def filter_func(model_response, right_answer):
    if "<think>" not in model_response["output_text"] \
        or "</think>" not in model_response["output_text"]:
        return False

    answer_token_index = retrieve_answer_token_index(model_response["score_data"])
    if answer_token_index is None or answer_token_index >= len(model_response["score_data"]) - 1:
        return False

    token_data = model_response["score_data"][answer_token_index]
    return right_answer in token_data["top_tokens"]

In [ ]:
# launching model

ERROR_LIMIT = 500

index = Index(f"../../../index_data/qwen2.5_RACE_reasoning_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring RACE",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ]
                }
            )
            
            response["dataset_elem"] = dataset[iter]

            if filter_func(response, str(ord(data["answer"]) - ord("A"))):
                if response["output_text"][-1] == str(
                    ord(data["answer"]) - ord("A")
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )
                
                response = {
                    "iteration": iter,
                    **response
                }

                index.save_data(response, iter, logging=False)
                break
            
            if error_count == ERROR_LIMIT:
                raise Exception("Error limit exceeded")
            
        except Exception as e:
            if (error_count + 1) % 500 == 0:
                print(f"Error on iteration {iter + 1}\nError count: {error_count}\nMessage: {e}")
            if error_count == ERROR_LIMIT:
                print("Error limit exceeded")
                

Файл не существует: ../../../index_data/qwen2.5-7B_RACE_reasoning_12000_index.pkl
Файл не существует: ../../../index_data/qwen2.5-7B_RACE_reasoning_12000_data.pkl
Index is cleared successfully.


Scoring RACE: 100%|██████████| 12000/12000 [11:08:03<00:00,  3.34s/it, Current accuracy=0.46225]              


In [12]:
print(len(index))

0
